In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from datetime import datetime, timedelta
import json

# All links in a list
links = [
    "https://www.theconnecticutscoop.com/statewide-news.html",
    "https://www.theconnecticutscoop.com/tolland-county.html",
    "https://www.theconnecticutscoop.com/windham-county.html",
    "https://www.theconnecticutscoop.com/hartford-county.html",
    "https://www.theconnecticutscoop.com/new-haven-county.html",
    "https://www.theconnecticutscoop.com/litchfield-county.html",
    "https://www.theconnecticutscoop.com/fairfield-county.html",
    "https://www.theconnecticutscoop.com/new-london-county.html",
    "https://www.theconnecticutscoop.com/middlesex-county.html",
    "https://www.theconnecticutscoop.com/massachusetts.html"
]

all_links = []

# ---- Date filter ----
today = datetime.today().date()
two_days_ago = today - timedelta(days=2)

# Create driver ONCE
driver = webdriver.Chrome()
wait = WebDriverWait(driver, 15)

for url in links:
    print(f"Scraping: {url}")
    driver.get(url)

    wait.until(
        EC.presence_of_all_elements_located(
            (By.CSS_SELECTOR, "div.waddons-blog-card")
        )
    )

    cards = driver.find_elements(By.CSS_SELECTOR, "div.waddons-blog-card")

    for card in cards:

        # ---- Heading ----
        heading_elements = card.find_elements(By.CSS_SELECTOR, "div.waddons-blog-header")
        heading = heading_elements[0].text.strip() if heading_elements else ""

        # ---- Date ----
        meta_elements = card.find_elements(By.CSS_SELECTOR, "div.waddons-blog-meta")
        article_date = None

        if meta_elements:
            meta_text = meta_elements[0].text.strip()
            date_text = meta_text.split("-")[0].strip()

            try:
                article_date = datetime.strptime(date_text, "%m/%d/%Y").date()
            except:
                article_date = None

        # ---- Link ----
        link_elements = card.find_elements(By.CSS_SELECTOR, "a.waddons-blog-card-link-full")
        link = link_elements[0].get_attribute("href") if link_elements else ""

        # ---- Filter last 2 days ----
        if link and article_date and article_date >= two_days_ago:
            print("Added:", heading, article_date)

            all_links.append({
                "heading": heading,
                "date": article_date,
                "link": link
            })

driver.quit()

# Remove duplicates
df = pd.DataFrame(all_links).drop_duplicates(subset=["link"])

# Sort newest first
df = df.sort_values(by="date", ascending=False)
 
print("Total articles found:", len(df))

# Save to Excel
export_path = f"ct_scoop_links_{datetime.now().strftime('%Y-%m-%d')}.xlsx"
df.to_excel(export_path, index=False)
print(f"Saved {len(df)} records to {export_path}")

# Save JSON for dashboard tab
json_path = "ct_scoop_latest.json"
json_payload = {
    "last_updated": today.strftime("%Y-%m-%d"),
    "data": df.to_dict(orient="records"),
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_payload, f, ensure_ascii=False, indent=2)
print(f"Saved JSON to {json_path}")

Scraping: https://www.theconnecticutscoop.com/statewide-news.html
Scraping: https://www.theconnecticutscoop.com/tolland-county.html
Added: UNION SCOOP: Travelers Restaurant becoming Blondie's 3rd eatery 2026-03-07
Added: STORRS SCOOP: Dog House Subs coming to Storrs Center 2026-03-06
Scraping: https://www.theconnecticutscoop.com/windham-county.html
Scraping: https://www.theconnecticutscoop.com/hartford-county.html
Added: BLOOMFIELD SCOOP: New development eyed for Bloomfield land 2026-03-07
Added: AVON SCOOP: Frootie Tooties opening new location in Avon 2026-03-07
Scraping: https://www.theconnecticutscoop.com/new-haven-county.html
Scraping: https://www.theconnecticutscoop.com/litchfield-county.html
Scraping: https://www.theconnecticutscoop.com/fairfield-county.html
Added: TRUMBULL SCOOP: New tenants coming soon to Trumbull Mall 2026-03-06
Scraping: https://www.theconnecticutscoop.com/new-london-county.html
Added: COLCHESTER SCOOP: Harry's Place opens for the season! 2026-03-07
Scraping:

In [ ]:
import anthropic
import requests
import re
import io
from bs4 import BeautifulSoup

# ---- 1. Fetch each article's full text ----
def fetch_article_text(url, timeout=15):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
    try:
        resp = requests.get(url, headers=headers, timeout=timeout)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        return soup.get_text(separator="
", strip=True)
    except Exception as e:
        return f"[Could not fetch: {e}]"

print("Fetching article content...")
articles = []
for _, row in df.iterrows():
    text = fetch_article_text(row["link"])
    articles.append({"heading": row["heading"], "link": row["link"], "text": text[:8000]})
    print(f"  Fetched: {row['heading'][:70]}")


# ---- 2. Build combined prompt ----
SYSTEM_PROMPT = """You are an expert, precise data extractor specialized in retail and restaurant openings and closures. I will provide multiple news articles (each usually starting with its source URL). For EVERY article, extract the following information strictly and only from the text provided — no assumptions, no external knowledge, no guessing zip codes, no inferring dates or statuses:

🔍 Extract these fields
• Store/Shop/Restaurant Name
• Location or Full Address with zip code (if no zip code is mentioned, write exactly the address given; if no address at all, write "Address not specified")
• Event Type (write exactly "Opening" or "Closing" based only on the article content)
• Event Date
  For openings → Opening Date
  For closures → Closing Date
  (write exact date or month/year if mentioned; otherwise write exactly "Not specified")
• Status
  For openings → use phrasing like: "under construction", "opening soon", "set to open", "recently opened", "grand opening on…", "planned for", etc.
  For closures → use phrasing like: "closed", "permanently closed", "closing soon", "set to close", "shut down", "liquidation", etc.
  👉 Use the exact phrasing or closest direct wording from the article — do NOT invent or normalize
• Short Description (exactly 2–3 concise sentences summarizing ONLY what the article says — no opinions, no extra context)

📊 Output format
Create ONE clean Markdown table with these exact column headers (in this order):

| Store/Shop/Restaurant Name | Location or Full Address with zip code | Event Type | Event Date | Status | Short Description | Article Link |

📌 Rules
- Add one row per article in the order the articles are given
- If an article contains multiple businesses, create a separate row for each
- If an article includes both openings and closures, extract each separately
- If an article has zero relevant business opening or closure information, still include a row with:
  * Store Name: "No qualifying business found"
  * Other columns: "N/A"

🚫 Strict constraints
❌ No assumptions
❌ No external data
❌ No inferred addresses or dates
❌ No rewriting or standardizing status text

📎 Final section (mandatory)
At the very end of your response, add:
Non-working or unusable articles List:
- Article numbers or URLs
- Reason (e.g. "paywall", "no business details", "duplicate", "text missing", etc.)"""

articles_body = ""
for i, art in enumerate(articles, 1):
    articles_body += f"

--- Article {i} ---
URL: {art['link']}
Heading: {art['heading']}

{art['text']}"


# ---- 3. Call Claude API (streaming) ----
print("
Calling Claude API...
")
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

full_response = ""
with client.messages.stream(
    model="claude-opus-4-6",
    max_tokens=8000,
    system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": articles_body}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
        full_response += text

print("

✅ Extraction complete.")


# ---- 4. Parse markdown table → DataFrame ----
def markdown_table_to_df(md_text):
    lines = md_text.strip().split("
")
    table_lines = [l for l in lines if l.strip().startswith("|")]
    if len(table_lines) < 2:
        print("⚠️  No table found in response.")
        return pd.DataFrame()
    headers = [c.strip() for c in table_lines[0].strip().strip("|").split("|")]
    data_lines = [l for l in table_lines[1:]
                  if not re.match(r"^\s*\|[\s\-|:]+\|\s*$", l)]
    rows = []
    for line in data_lines:
        cells = [c.strip() for c in line.strip().strip("|").split("|")]
        while len(cells) < len(headers):
            cells.append("")
        rows.append(cells[:len(headers)])
    return pd.DataFrame(rows, columns=headers)

result_df = markdown_table_to_df(full_response)
print(f"
Extracted {len(result_df)} row(s).")
display(result_df)


# ---- 5. Save to CSV ----
csv_path = f"ct_scoop_extracted_{datetime.now().strftime('%Y-%m-%d')}.csv"
result_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"
💾 Saved to: {csv_path}")


In [ ]:
today = datetime.today()
two_days_ago = today - timedelta(days=2)

print(today)

2026-03-08 01:33:59.652152
